# 02 — Data Cleaning and Feature Engineering

Clean raw taxi data, standardize columns, filter invalid records, create features for analysis.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

## Project Paths

In [2]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
docs_dir = project_root / "docs"

processed_data_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

trip_data_path = raw_data_dir / "yellow_tripdata_2024-01.parquet"
zone_lookup_path = raw_data_dir / "taxi_zone_lookup.csv"

cleaned_trip_output_path = processed_data_dir / "cleaned_yellow_taxi_jan_2024.parquet"
cleaning_log_output_path = processed_data_dir / "cleaning_log.csv"

print(f"Project root: {project_root}")
print(f"Raw trip data path: {trip_data_path}")
print(f"Zone lookup path: {zone_lookup_path}")

Project root: /Users/angelonelson/Projects/NYCTaxiTripAnalytics
Raw trip data path: /Users/angelonelson/Projects/NYCTaxiTripAnalytics/data/raw/yellow_tripdata_2024-01.parquet
Zone lookup path: /Users/angelonelson/Projects/NYCTaxiTripAnalytics/data/raw/taxi_zone_lookup.csv


## Load Raw Data

In [3]:
taxi_trips = pd.read_parquet(trip_data_path)
taxi_zones = pd.read_csv(zone_lookup_path)

print(f"Raw taxi trips shape: {taxi_trips.shape[0]:,} rows and {taxi_trips.shape[1]:,} columns")
print(f"Taxi zone lookup shape: {taxi_zones.shape[0]:,} rows and {taxi_zones.shape[1]:,} columns")

Raw taxi trips shape: 2,964,624 rows and 19 columns
Taxi zone lookup shape: 265 rows and 4 columns


## Cleaning Log Setup

In [4]:
cleaning_log = []

def add_cleaning_log(step_name, rows_before, rows_after, description):
    rows_removed = rows_before - rows_after
    removal_percentage = (rows_removed / rows_before * 100) if rows_before else 0

    cleaning_log.append({
        "step_name": step_name,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_removed,
        "removal_percentage": round(removal_percentage, 4),
        "description": description,
    })

## Standardize Column Names

In [5]:
trip_column_mapping = {
    "VendorID": "vendor_id",
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "passenger_count": "passenger_count",
    "trip_distance": "trip_distance",
    "RatecodeID": "rate_code_id",
    "store_and_fwd_flag": "store_and_fwd_flag",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
    "payment_type": "payment_type",
    "fare_amount": "fare_amount",
    "extra": "extra",
    "mta_tax": "mta_tax",
    "tip_amount": "tip_amount",
    "tolls_amount": "tolls_amount",
    "improvement_surcharge": "improvement_surcharge",
    "total_amount": "total_amount",
    "congestion_surcharge": "congestion_surcharge",
    "Airport_fee": "airport_fee",
}

zone_column_mapping = {
    "LocationID": "location_id",
    "Borough": "borough",
    "Zone": "zone",
    "service_zone": "service_zone",
}

taxi_trips = taxi_trips.rename(columns=trip_column_mapping)
taxi_zones = taxi_zones.rename(columns=zone_column_mapping)

print("Taxi trip columns:")
print(taxi_trips.columns.tolist())

print("\nTaxi zone lookup columns:")
print(taxi_zones.columns.tolist())

Taxi trip columns:
['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'rate_code_id', 'store_and_fwd_flag', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']

Taxi zone lookup columns:
['location_id', 'borough', 'zone', 'service_zone']


## Remove Duplicates

In [6]:
rows_before = len(taxi_trips)

taxi_trips = taxi_trips.drop_duplicates().copy()

rows_after = len(taxi_trips)

add_cleaning_log(
    step_name="remove_exact_duplicates",
    rows_before=rows_before,
    rows_after=rows_after,
    description="Removed exact duplicate trip records."
)

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal: {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")

Rows before duplicate removal: 2,964,624
Rows after duplicate removal: 2,964,624
Rows removed: 0


## Convert Timestamps

In [7]:
taxi_trips["pickup_datetime"] = pd.to_datetime(taxi_trips["pickup_datetime"], errors="coerce")
taxi_trips["dropoff_datetime"] = pd.to_datetime(taxi_trips["dropoff_datetime"], errors="coerce")

print(taxi_trips[["pickup_datetime", "dropoff_datetime"]].dtypes)

pickup_datetime     datetime64[us]
dropoff_datetime    datetime64[us]
dtype: object


## Remove Invalid Timestamps

In [8]:
rows_before = len(taxi_trips)

taxi_trips = taxi_trips[
    taxi_trips["pickup_datetime"].notna()
    & taxi_trips["dropoff_datetime"].notna()
    & (taxi_trips["dropoff_datetime"] > taxi_trips["pickup_datetime"])
].copy()

rows_after = len(taxi_trips)

add_cleaning_log(
    step_name="remove_invalid_timestamps",
    rows_before=rows_before,
    rows_after=rows_after,
    description="Removed records with missing timestamps or drop-off times earlier than pickup times."
)

print(f"Rows after timestamp validation: {rows_after:,}")

Rows after timestamp validation: 2,963,754


## Keep Only January 2024

In [9]:
rows_before = len(taxi_trips)

january_start = pd.Timestamp("2024-01-01 00:00:00")
february_start = pd.Timestamp("2024-02-01 00:00:00")

taxi_trips = taxi_trips[
    (taxi_trips["pickup_datetime"] >= january_start)
    & (taxi_trips["pickup_datetime"] < february_start)
].copy()

rows_after = len(taxi_trips)

add_cleaning_log(
    step_name="restrict_to_january_2024_pickups",
    rows_before=rows_before,
    rows_after=rows_after,
    description="Kept only trips with pickup timestamps in January 2024."
)

print(f"Rows after January 2024 filtering: {rows_after:,}")

Rows after January 2024 filtering: 2,963,736


## Trip Duration

In [10]:
taxi_trips["trip_duration_minutes"] = (
    taxi_trips["dropoff_datetime"] - taxi_trips["pickup_datetime"]
).dt.total_seconds() / 60

taxi_trips[["pickup_datetime", "dropoff_datetime", "trip_duration_minutes"]].head()

,pickup_datetime,dropoff_datetime,trip_duration_minutes
0,2024-01-01 00:57:55,2024-01-01 01:17:43,19.80
1,2024-01-01 00:03:00,2024-01-01 00:09:36,6.60
2,2024-01-01 00:17:06,2024-01-01 00:35:01,17.92
3,2024-01-01 00:36:38,2024-01-01 00:44:56,8.30
4,2024-01-01 00:46:51,2024-01-01 00:52:57,6.10


## Remove Impossible Trips

In [11]:
rows_before = len(taxi_trips)

taxi_trips = taxi_trips[
    (taxi_trips["trip_distance"] > 0)
    & (taxi_trips["trip_duration_minutes"] > 0)
    & (taxi_trips["trip_duration_minutes"] <= 180)
    & (taxi_trips["fare_amount"] >= 0)
    & (taxi_trips["total_amount"] >= 0)
].copy()

rows_after = len(taxi_trips)

add_cleaning_log(
    step_name="remove_impossible_trip_values",
    rows_before=rows_before,
    rows_after=rows_after,
    description="Removed trips with impossible distance, duration, fare, or total amount values."
)

print(f"Rows after impossible value filtering: {rows_after:,}")

Rows after impossible value filtering: 2,868,035


## Handle Passenger Count

In [12]:
taxi_trips["passenger_count_clean"] = taxi_trips["passenger_count"]

taxi_trips.loc[
    (taxi_trips["passenger_count_clean"] < 0) | (taxi_trips["passenger_count_clean"] > 6),
    "passenger_count_clean"
] = np.nan

taxi_trips["passenger_count_group"] = np.select(
    [
        taxi_trips["passenger_count_clean"].isna(),
        taxi_trips["passenger_count_clean"] == 0,
        taxi_trips["passenger_count_clean"] == 1,
        taxi_trips["passenger_count_clean"].between(2, 3),
        taxi_trips["passenger_count_clean"].between(4, 6),
    ],
    [
        "Unknown",
        "Zero reported",
        "Solo",
        "Small group",
        "Large group",
    ],
    default="Unknown"
)

taxi_trips["passenger_count_clean"] = taxi_trips["passenger_count_clean"].fillna(0).astype(int)

taxi_trips["passenger_count_group"].value_counts(dropna=False)

passenger_count_group
Solo             2133482
Small group       483808
Unknown           115280
Large group       104788
Zero reported      30677
Name: count, dtype: int64

## Readable Labels for Encoded Fields

In [13]:
vendor_mapping = {
    1: "Creative Mobile Technologies",
    2: "VeriFone Inc.",
}

rate_code_mapping = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester",
    5: "Negotiated fare",
    6: "Group ride",
    99: "Unknown",
}

payment_type_mapping = {
    0: "Unknown",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}

taxi_trips["vendor_name"] = taxi_trips["vendor_id"].map(vendor_mapping).fillna("Unknown")
taxi_trips["rate_code_label"] = taxi_trips["rate_code_id"].map(rate_code_mapping).fillna("Unknown")
taxi_trips["payment_type_label"] = taxi_trips["payment_type"].map(payment_type_mapping).fillna("Unknown")
taxi_trips["store_and_fwd_flag"] = taxi_trips["store_and_fwd_flag"].fillna("Unknown")

taxi_trips[["vendor_id", "vendor_name", "rate_code_id", "rate_code_label", "payment_type", "payment_type_label"]].head()

,vendor_id,vendor_name,rate_code_id,rate_code_label,payment_type,payment_type_label
0,2,VeriFone Inc.,1.00,Standard rate,2,Cash
1,1,Creative Mobile Technologies,1.00,Standard rate,1,Credit card
2,1,Creative Mobile Technologies,1.00,Standard rate,1,Credit card
3,1,Creative Mobile Technologies,1.00,Standard rate,1,Credit card
4,1,Creative Mobile Technologies,1.00,Standard rate,1,Credit card


## Time-Based Features

In [14]:
taxi_trips["pickup_date"] = taxi_trips["pickup_datetime"].dt.date
taxi_trips["pickup_year"] = taxi_trips["pickup_datetime"].dt.year
taxi_trips["pickup_month"] = taxi_trips["pickup_datetime"].dt.month
taxi_trips["pickup_day"] = taxi_trips["pickup_datetime"].dt.day
taxi_trips["pickup_hour"] = taxi_trips["pickup_datetime"].dt.hour
taxi_trips["pickup_day_name"] = taxi_trips["pickup_datetime"].dt.day_name()
taxi_trips["pickup_day_of_week"] = taxi_trips["pickup_datetime"].dt.dayofweek
taxi_trips["is_weekend"] = taxi_trips["pickup_day_of_week"].isin([5, 6])

taxi_trips[[
    "pickup_datetime",
    "pickup_date",
    "pickup_hour",
    "pickup_day_name",
    "is_weekend"
]].head()

,pickup_datetime,pickup_date,pickup_hour,pickup_day_name,is_weekend
0,2024-01-01 00:57:55,2024-01-01,0,Monday,False
1,2024-01-01 00:03:00,2024-01-01,0,Monday,False
2,2024-01-01 00:17:06,2024-01-01,0,Monday,False
3,2024-01-01 00:36:38,2024-01-01,0,Monday,False
4,2024-01-01 00:46:51,2024-01-01,0,Monday,False


## Business Metrics

In [15]:
taxi_trips["revenue_per_mile"] = np.where(
    taxi_trips["trip_distance"] > 0,
    taxi_trips["total_amount"] / taxi_trips["trip_distance"],
    np.nan
)

taxi_trips["fare_per_minute"] = np.where(
    taxi_trips["trip_duration_minutes"] > 0,
    taxi_trips["fare_amount"] / taxi_trips["trip_duration_minutes"],
    np.nan
)

taxi_trips["tip_percentage"] = np.where(
    taxi_trips["fare_amount"] > 0,
    taxi_trips["tip_amount"] / taxi_trips["fare_amount"] * 100,
    0
)

taxi_trips["distance_bucket"] = pd.cut(
    taxi_trips["trip_distance"],
    bins=[0, 1, 3, 5, 10, 20, np.inf],
    labels=["0-1 miles", "1-3 miles", "3-5 miles", "5-10 miles", "10-20 miles", "20+ miles"],
    include_lowest=True
)

taxi_trips["duration_bucket"] = pd.cut(
    taxi_trips["trip_duration_minutes"],
    bins=[0, 5, 10, 20, 30, 60, np.inf],
    labels=["0-5 min", "5-10 min", "10-20 min", "20-30 min", "30-60 min", "60+ min"],
    include_lowest=True
)

taxi_trips[[
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "revenue_per_mile",
    "fare_per_minute",
    "tip_percentage",
    "distance_bucket",
    "duration_bucket",
]].head()

,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount,revenue_per_mile,fare_per_minute,tip_percentage,distance_bucket,duration_bucket
0,1.72,19.80,17.70,0.00,22.70,13.20,0.89,0.00,1-3 miles,10-20 min
1,1.80,6.60,10.00,3.75,18.75,10.42,1.52,37.50,1-3 miles,5-10 min
2,4.70,17.92,23.30,3.00,31.30,6.66,1.30,12.88,3-5 miles,10-20 min
3,1.40,8.30,10.00,2.00,17.00,12.14,1.20,20.00,1-3 miles,5-10 min
4,0.80,6.10,7.90,3.20,16.10,20.12,1.30,40.51,0-1 miles,5-10 min


## Join Zone Lookup

In [16]:
pickup_zones = taxi_zones.rename(columns={
    "location_id": "pickup_location_id",
    "borough": "pickup_borough",
    "zone": "pickup_zone",
    "service_zone": "pickup_service_zone",
})

dropoff_zones = taxi_zones.rename(columns={
    "location_id": "dropoff_location_id",
    "borough": "dropoff_borough",
    "zone": "dropoff_zone",
    "service_zone": "dropoff_service_zone",
})

taxi_trips = taxi_trips.merge(
    pickup_zones,
    on="pickup_location_id",
    how="left"
)

taxi_trips = taxi_trips.merge(
    dropoff_zones,
    on="dropoff_location_id",
    how="left"
)

location_columns = [
    "pickup_location_id",
    "pickup_borough",
    "pickup_zone",
    "dropoff_location_id",
    "dropoff_borough",
    "dropoff_zone",
]

taxi_trips[location_columns].head()

,pickup_location_id,pickup_borough,pickup_zone,dropoff_location_id,dropoff_borough,dropoff_zone
0,186,Manhattan,Penn Station/Madison Sq West,79,Manhattan,East Village
1,140,Manhattan,Lenox Hill East,236,Manhattan,Upper East Side North
2,236,Manhattan,Upper East Side North,79,Manhattan,East Village
3,79,Manhattan,East Village,211,Manhattan,SoHo
4,211,Manhattan,SoHo,148,Manhattan,Lower East Side


## Fill Missing Location Labels

In [17]:
location_label_columns = [
    "pickup_borough",
    "pickup_zone",
    "pickup_service_zone",
    "dropoff_borough",
    "dropoff_zone",
    "dropoff_service_zone",
]

for column_name in location_label_columns:
    taxi_trips[column_name] = taxi_trips[column_name].fillna("Unknown")

taxi_trips[location_label_columns].isna().sum()

pickup_borough          0
pickup_zone             0
pickup_service_zone     0
dropoff_borough         0
dropoff_zone            0
dropoff_service_zone    0
dtype: int64

## Outlier Flags

In [18]:
total_amount_99th_percentile = taxi_trips["total_amount"].quantile(0.99)
trip_distance_99th_percentile = taxi_trips["trip_distance"].quantile(0.99)
trip_duration_99th_percentile = taxi_trips["trip_duration_minutes"].quantile(0.99)

taxi_trips["is_high_value_trip"] = taxi_trips["total_amount"] > total_amount_99th_percentile
taxi_trips["is_long_distance_trip"] = taxi_trips["trip_distance"] > trip_distance_99th_percentile
taxi_trips["is_long_duration_trip"] = taxi_trips["trip_duration_minutes"] > trip_duration_99th_percentile

print(f"99th percentile total amount: {total_amount_99th_percentile:.2f}")
print(f"99th percentile trip distance: {trip_distance_99th_percentile:.2f}")
print(f"99th percentile trip duration: {trip_duration_99th_percentile:.2f}")

99th percentile total amount: 103.13
99th percentile trip distance: 20.03
99th percentile trip duration: 59.72


## Final Cleaned Dataset Review

In [19]:
print(f"Cleaned dataset shape: {taxi_trips.shape[0]:,} rows and {taxi_trips.shape[1]:,} columns")

cleaned_missing_summary = (
    taxi_trips
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column_name", 0: "missing_count"})
)

cleaned_missing_summary["missing_percentage"] = (
    cleaned_missing_summary["missing_count"] / len(taxi_trips) * 100
).round(2)

cleaned_missing_summary.sort_values("missing_percentage", ascending=False).head(20)

Cleaned dataset shape: 2,868,035 rows and 47 columns


,column_name,missing_count,missing_percentage
3,passenger_count,115237,4.02
5,rate_code_id,115237,4.02
18,airport_fee,115237,4.02
17,congestion_surcharge,115237,4.02
0,vendor_id,0,0.00
35,tip_percentage,0,0.00
28,pickup_day,0,0.00
29,pickup_hour,0,0.00
30,pickup_day_name,0,0.00
31,pickup_day_of_week,0,0.00


## Save Cleaned Dataset

In [20]:
taxi_trips.to_parquet(cleaned_trip_output_path, index=False)

cleaning_log_dataframe = pd.DataFrame(cleaning_log)
cleaning_log_dataframe.to_csv(cleaning_log_output_path, index=False)

cleaned_missing_summary.to_csv(processed_data_dir / "cleaned_missing_value_summary.csv", index=False)

cleaning_summary_markdown = f"""# Cleaning Summary

## Input Dataset

- Raw file: yellow_tripdata_2024-01.parquet
- Initial rows: {cleaning_log_dataframe.iloc[0]["rows_before"]:,}

## Output Dataset

- Cleaned file: cleaned_yellow_taxi_jan_2024.parquet
- Final rows: {len(taxi_trips):,}
- Final columns: {taxi_trips.shape[1]:,}

## Cleaning Steps

{cleaning_log_dataframe.to_markdown(index=False)}

## Major Features Created

- trip_duration_minutes
- pickup_date
- pickup_hour
- pickup_day_name
- is_weekend
- revenue_per_mile
- fare_per_minute
- tip_percentage
- distance_bucket
- duration_bucket
- vendor_name
- rate_code_label
- payment_type_label
- pickup_borough
- pickup_zone
- dropoff_borough
- dropoff_zone
- is_high_value_trip
- is_long_distance_trip
- is_long_duration_trip

## Notes

The raw dataset was not modified. All cleaning was performed through a reproducible Python workflow.
"""

with open(docs_dir / "cleaning_summary.md", "w", encoding="utf-8") as file:
    file.write(cleaning_summary_markdown)

print(f"Cleaned dataset saved to: {cleaned_trip_output_path}")
print(f"Cleaning log saved to: {cleaning_log_output_path}")
print("Cleaning summary saved to docs/cleaning_summary.md")

Cleaned dataset saved to: /Users/angelonelson/Projects/NYCTaxiTripAnalytics/data/processed/cleaned_yellow_taxi_jan_2024.parquet
Cleaning log saved to: /Users/angelonelson/Projects/NYCTaxiTripAnalytics/data/processed/cleaning_log.csv
Cleaning summary saved to docs/cleaning_summary.md


## Done

Cleaning complete. Next: `03_eda.ipynb`.